In [1]:
import json
import subprocess
import signal
from dotenv import load_dotenv
import requests

In [ ]:
# loading .env
load_dotenv('.env')

In [3]:
def list_model_ids():
  url = "http://localhost:8000/v1/models"
  response = requests.get(url)
  return [model['id'] for model in response.json()['data']]

In [4]:
def stop_vllm_server():
    """
    Stop a background vLLM server cleanly (SIGTERM).
    Falls back to SIGKILL if needed.
    """
    try:
        # Find vLLM server PIDs
        result = subprocess.check_output(
            ["pgrep", "-f", "vllm serve"],
            text=True
        ).strip()

        if not result:
            print("No vLLM server running.")
            return

        pids = result.splitlines()
        for pid in pids:
            print(f"Stopping vLLM server (PID {pid})...")
            subprocess.run(["kill", pid])

    except subprocess.CalledProcessError:
        print("No vLLM server running.")

In [5]:
def get_response(system_prompt, prompt, model, temperature=0.1, max_tokens=128):

  url = "https://ew388nf9hgny79-8000.proxy.runpod.net/v1/chat/completions"

  payload = {
      "model": model,
      "messages": [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": prompt}
      ],
      "temperature": temperature,
      "max_tokens": max_tokens,
  }

  response = requests.post(url, json=payload)
  return response.json()['choices'][0]['message']['content']


In [6]:
SYSTEM_PROMPT = "You are a helpful AI assistant specialised in African history which gives concise answers to questions asked."
TEST_PROMPTS = [
    "Who was the richest king in history from the Mali Empire?",
    "Explain the significance of the Great Zimbabwe ruins.",
    "Describe the impact of the Ashanti Empire on West African trade.",
    "What was the role of the Kingdom of Aksum in early Christianity?",
    "Detail the leadership of Queen Amina of Zazzau."
]

In [7]:
filename = "history_messages_dataset.jsonl"
total_requests = 125

In [ ]:
# Formating the TEST_PROMPTS FOR CUSTOM BENCHMARKING PROCESS IN VLLM
with open(filename, "w") as f:
    for i in range (total_requests):
        user_q = TEST_PROMPTS[i % len(TEST_PROMPTS)]
        full_prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{user_q}\n<|assistant|>\n"
        data = {
            "prompt":full_prompt,
            # This acts as a hint for the benchmark 
            # to know how many tokens to expect in the output, 
            # so that it can wait for the complete response 
            # before stopping the timer.
            "output_len": 128
        }
        f.write(json.dumps(data) + "\n")

print(f"✅ Created the {filename} file with {total_requests} requests.")

✅ Created the history_messages_dataset.jsonl file with 125 requests.


In [9]:
# !nohup vllm serve microsoft/Phi-4-mini-instruct \
#     --enable-lora \
#     --lora-modules african-history=DannyAI/phi4_african_history_lora_ds2_axolotl \
#     --max-model-len 512 \
#     # --enable-metrics True \
#     # --host my_server_ip \
#     # --port 7860 \
#     --gpu-memory-utilization 0.9 \
#     > vllm.log 2>&1 &

### **Errors Faced**

In [ ]:
# ---------------------------------------------------------------------------
# OSError                                   Traceback (most recent call last)
# Cell In[9], line 1
# ----> 1 get_ipython().system('vllm serve microsoft/Phi-4-mini-instruct      --enable-lora      --lora-modules african-history=DannyAI/phi4_african_history_lora_ds2_axolotl      -- tokenizer microsoft/Phi-4-mini-instruct      --max-model-len 1024      --gpu-memory-utilization 0.9      > vllm.log 2>&1 &')

# File ~\ipykernel\zmqshell.py:772, in ZMQInteractiveShell.system_piped(self, cmd)
#     765 if cmd.rstrip().endswith("&"):
#     766     # this is *far* from a rigorous test
#     767     # We do not support backgrounding processes because we either use
#     768     # pexpect or pipes to read from.  Users can always just call
#     769     # os.system() or use ip.system=ip.system_raw
#     770     # if they really want a background process.
#     771     msg = "Background processes not supported."
# --> 772     raise OSError(msg)
#     774 # we explicitly do NOT return the subprocess status code, because
#     775 # a non-None value would trigger :func:`sys.displayhook` calls.
#     776 # Instead, we store the exit_code in user_ns.
#     777 # Also, protect system call from UNC paths on Windows here too
#     778 # as is done in InteractiveShell.system_raw
#     779 if sys.platform == "win32":

# OSError: Background processes not supported.

### **Due to this error, Runpod must be used**

In [ ]:
# ---------------------------------------------------------------------------
# OSError                                   Traceback (most recent call last)
# Cell In[14], line 1
# ----> 1 get_ipython().system('nohup vllm serve microsoft/Phi-4-mini-instruct      --enable-lora      --lora-modules african-history=DannyAI/phi4_african_history_lora_ds2_axolotl      --max-model-len 512      # --enable-metrics True      # --host my_server_ip      # --port 7860      --gpu-memory-utilization 0.9      > vllm.log 2>&1 &')

# File /usr/local/lib/python3.11/dist-packages/ipykernel/zmqshell.py:641, in ZMQInteractiveShell.system_piped(self, cmd)
#     634 if cmd.rstrip().endswith("&"):
#     635     # this is *far* from a rigorous test
#     636     # We do not support backgrounding processes because we either use
#     637     # pexpect or pipes to read from.  Users can always just call
#     638     # os.system() or use ip.system=ip.system_raw
#     639     # if they really want a background process.
#     640     msg = "Background processes not supported."
# --> 641     raise OSError(msg)
#     643 # we explicitly do NOT return the subprocess status code, because
#     644 # a non-None value would trigger :func:`sys.displayhook` calls.
#     645 # Instead, we store the exit_code in user_ns.
#     646 # Also, protect system call from UNC paths on Windows here too
#     647 # as is done in InteractiveShell.system_raw
#     648 if sys.platform == "win32":

# OSError: Background processes not supported.

In [ ]:
# Command line (CMD) had to be used in runpod due to this error

In [ ]:
# Copy and paste this in the terminal
vllm serve microsoft/Phi-4-mini-instruct \
    --enable-lora \
    --lora-modules african-history=DannyAI/phi4_african_history_lora_ds2_axolotl \
    --max-model-len 512 \
    --gpu-memory-utilization 0.9 \
    # > vllm.log 2>&1 &

In [12]:
while True:
  try:
    list_model_ids()
    break
  except:
    pass

system_prompt  = "You are a helpful AI assistant specialised in African history which gives concise answers to questions asked."
prompt = "Who was the richest king in history from the Mali Empire?"

model_ids = list_model_ids()
base_model_id = model_ids[0]
lora_model_id = model_ids[1]

base_response = get_response(system_prompt, prompt, base_model_id)
lora_response = get_response(system_prompt, prompt, lora_model_id)

In [13]:
print(base_response)
print(lora_response)

Mansa Musa
Mansa Musa


In [14]:
import requests
import json

def stream_response(system_prompt, prompt, model, temperature=0.1, max_tokens=150):
    url = "https://ew388nf9hgny79-8000.proxy.runpod.net/v1/chat/completions"

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": True,
    }

    with requests.post(url, json=payload, stream=True) as response:
        response.raise_for_status()

        for line in response.iter_lines():
            if not line:
                continue

            # vLLM uses SSE-style "data: ..."
            decoded = line.decode("utf-8")
            if decoded.startswith("data: "):
                data = decoded[len("data: "):]

                if data == "[DONE]":
                    break

                chunk = json.loads(data)
                delta = chunk["choices"][0]["delta"]

                if "content" in delta:
                    yield delta["content"]


In [15]:
print(base_model_id)
print(lora_model_id)

microsoft/Phi-4-mini-instruct
african-history


In [16]:
for token in stream_response(
    system_prompt  = system_prompt,
    prompt = prompt,
    # model=base_model_id,
    model=lora_model_id,
):
    print(token, end="", flush=True)


Mansa Musa

In [17]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

start_time = time.time()

def fire_requests_concurrently(
    system_prompt,
    prompt,
    model,
    temperature=0.1,
    max_tokens=128,
    n_requests=10
):
    """
    Fire multiple requests at the same time and return all responses.
    """
    responses = []

    with ThreadPoolExecutor(max_workers=n_requests) as executor:
        futures = [
            executor.submit(
                get_response,
                system_prompt,
                prompt,
                model,
                temperature,
                max_tokens,
            )
            for _ in range(n_requests)
        ]

        for future in as_completed(futures):
            responses.append(future.result())

    return responses


outputs = fire_requests_concurrently(
    system_prompt=system_prompt,
    prompt=prompt,
    # model=base_model_id,
    model=lora_model_id,
    n_requests=40,
    temperature=0.7
)



for i, out in enumerate(outputs, 1):
    print(f"Response {i}:\n{out}\n")

end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken: {total_time} seconds")


Response 1:
Mansa Musa

Response 2:
Mansa Musa

Response 3:
Mansa Musa.

Response 4:
Mansa Musa

Response 5:
Mansa Musa

Response 6:
Mansa Musa

Response 7:
Mansa Musa

Response 8:
Mansa Musa.

Response 9:
Mansa Musa

Response 10:
Mansa Musa

Response 11:
Mansa Musa

Response 12:
Mansa Musa

Response 13:
Mansa Musa

Response 14:
Mansa Musa

Response 15:
Mansa Musa

Response 16:
Mansa Musa

Response 17:
Mansa Musa I

Response 18:
Mansa Musa

Response 19:
Mansa Musa

Response 20:
Mansa Musa

Response 21:
Mansa Musa

Response 22:
Mansa Musa I

Response 23:
Mansa Musa.

Response 24:
Mansa Musa I.

Response 25:
Mansa Musa I

Response 26:
Mansa Musa.

Response 27:
Mansa Musa.

Response 28:
Mansa Musa I.

Response 29:
Sundiata Keita

Response 30:
Sundiata Keita

Response 31:
Sundiata Keita.

Response 32:
Mansa Musa was the richest king in history from the Mali Empire.

Response 33:
Mansa Musa, the tenth Mansa of the Mali Empire.

Response 34:
Mansa Musa I is considered the richest king in his

In [ ]:
# # For killing the server
# stop_vllm_server()

## **Benchmarking TESTS**

In [ ]:
# 4 concurrecncy
vllm bench serve \
    --model african-history \
    --tokenizer microsoft/Phi-4-mini-instruct \
    --dataset-name custom \
    --dataset-path ./data/history_messages_dataset.jsonl \
    --num-prompts 125 \
    --max-concurrency 4 \
    --metric-percentiles 95,99 \
    --save-result \
    --result-dir ./results_african_history/run_c4

In [ ]:
# 16 concurrecncy
vllm bench serve \
    --model african-history \
    --tokenizer microsoft/Phi-4-mini-instruct \
    --dataset-name custom \
    --dataset-path ./data/history_messages_dataset.jsonl \
    --num-prompts 125 \
    --max-concurrency 16 \
    --metric-percentiles 95,99 \
    --save-result \
    --result-dir ./results_african_history/run_c16

In [ ]:
# 32 concurrecncy
vllm bench serve \
    --model african-history \
    --tokenizer microsoft/Phi-4-mini-instruct \
    --dataset-name custom \
    --dataset-path ./data/history_messages_dataset.jsonl \
    --num-prompts 125 \
    --max-concurrency 32 \
    --metric-percentiles 95,99 \
    --save-result \
    --result-dir ./results_african_history/run_c32

In [ ]:
# 64 concurrecncy
vllm bench serve \
    --model african-history \
    --tokenizer microsoft/Phi-4-mini-instruct \
    --dataset-name custom \
    --dataset-path ./data/history_messages_dataset.jsonl \
    --num-prompts 125 \
    --max-concurrency 64 \
    --metric-percentiles 95,99 \
    --save-result \
    --result-dir ./results_african_history/run_c64

### **Deriving E2E Metrics**

In [39]:
import json
import glob
import os
import pandas as pd

def get_vllm_e2e_metrics(result_dir):
    search_path = os.path.join(result_dir, "*.json")
    files = glob.glob(search_path)
    
    if not files:
        return pd.DataFrame()
    
    latest_file = max(files, key=os.path.getctime)
    
    with open(latest_file, 'r') as f:
        data = json.load(f)

    # 1. Calculate Average Tokens per Request
    completed = data.get('completed', 1)
    total_out = data.get('total_output_tokens', 0)
    avg_tokens = (total_out / completed) if completed > 0 else 0
    # We subtract 1 because the first token is covered by TTFT
    decode_tokens = max(0, avg_tokens - 1)

    # 2. Derive E2E metrics from component percentiles (ms)
    # E2E = TTFT + (TPOT * Decode_Tokens)
    def calc_e2e(ttft_key, tpot_key):
        ttft = data.get(ttft_key, 0)
        tpot = data.get(tpot_key, 0)
        return ttft + (tpot * decode_tokens)

    metrics = {
        "Mean":   calc_e2e('mean_ttft_ms', 'mean_tpot_ms'),
        "Median": calc_e2e('median_ttft_ms', 'median_tpot_ms'),
        "P95":    calc_e2e('p95_ttft_ms', 'p95_tpot_ms'),
        "P99":    calc_e2e('p99_ttft_ms', 'p99_tpot_ms'),
        # "TPS":    data.get('output_throughput', 0)
    }
    
    df = pd.DataFrame([metrics], index=[os.path.basename(result_dir)])
    return df.round(4)

In [40]:
# --- Execution ---
directories = [
    "./results_african_history/run_c4",
    "./results_african_history/run_c16",
    "./results_african_history/run_c32",
    "./results_african_history/run_c64"
]

all_results = [get_vllm_e2e_metrics(d) for d in directories if os.path.exists(d)]
all_results = [res for res in all_results if not res.empty]

if all_results:
    master_table = pd.concat(all_results)
    master_table.index.name = "Concurrency"
    display(master_table)
else:
    print("No data found in directories.")

,Mean,Median,P95,P99
Concurrency,,,,
run_c4,1536.8352,1538.7847,1557.7486,1565.1866
run_c16,1650.0509,1653.4975,1712.0464,1720.1676
run_c32,1806.7935,1818.5273,1884.9539,1901.2364
run_c64,2030.4288,2065.3881,2157.9292,2177.1231
